```sql
03 - Feature Iteration

Goal:
- start from the baseline setup
- add small feature groups one by one
- compare each iteration against the same validation split

This notebook is for feature testing, not for model tuning.
```

In [3]:
import pathlib as Path

In [4]:
import pandas as pd
import numpy as np

In [5]:
from lightgbm import LGBMClassifier

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score

#### Import Data

In [7]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [8]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [9]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [10]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [11]:
available_features = [c for c in base_features if c in train_df.columns]

In [12]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [13]:
new_features = ["log_txnamt", "has_identity"]

In [14]:
feature_cols=available_features+new_features

#### Model Training and Result Function

In [15]:
def model_input_data(train_df,valid_df,feature_cols,target_col):

    #Splitting Target vs Predictors
    X_train=train_df[feature_cols].copy()
    X_valid=valid_df[feature_cols].copy()

    y_train=train_df[target_col].copy()
    y_valid=train_df[target_col].copy()

    #Seperating Categorical and Numerical Columns
    cat_cols=X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    num_cols=X_train.select_dtypes(include="number").columns.tolist()

    #Filling Missing Values
    for col in num_cols:
        X_train[col]=X_train[col].fillna(X_train[col].median())
        X_valid[col]=X_valid[col].fillna(X_valid[col].median())

    return X_train, X_valid, y_train, y_valid, cat_cols